# 06 Local-Region Expert MoE x DQA Fifteen Loops

This notebook tests the next MoE idea:

> Do not mix models at the client level.  Grow experts around local
> pseudo-GT regions that are actually learnable.

This is a fast design/evidence loop, not fifteen full YOLO trainings.
It uses the completed `03_main_bn_residual_dqa_experiment` and the
previous MoE router results to rank fifteen local-region expert
hypotheses, then writes the concrete next full experiment candidate.

## Research Seeds

- PSSFL/FedMox: spatial router + Soft-Mixture for practical
  semi-supervised federated object detection.
- Soft MoE: differentiable soft assignment instead of brittle hard
  token routing.
- Expert Choice Routing: experts choose fixed-capacity tokens, which
  maps naturally to pseudo-GT quota control.
- MMoE / PLE: shared-private experts for task/domain relationship
  modeling.
- DAMEX: detection can benefit from dataset/domain-aware MoE, but
  routing should avoid expert collapse.

In [1]:
from pathlib import Path
import subprocess
import sys

import pandas as pd

cwd = Path.cwd().resolve()
if cwd.name == "notebooks" and cwd.parent.name == "moe":
    MOE_ROOT = cwd.parent
elif (cwd / "dynamic_quality_aware_classwise_aggregation").exists():
    MOE_ROOT = cwd / "dynamic_quality_aware_classwise_aggregation" / "scene_daynight_dqa" / "moe"
else:
    MOE_ROOT = cwd

SCENE_ROOT = MOE_ROOT.parent
WORKSPACE = MOE_ROOT / "output" / "06_spatial_expert_fifteen_loops"
SOURCE_WORKSPACE = SCENE_ROOT / "output" / "03_main_bn_residual_dqa_experiment"
PREV_MOE_WORKSPACE = MOE_ROOT / "output" / "05_router_ten_loops"
RUNNER = MOE_ROOT / "scripts" / "run_moe_06_spatial_expert_fifteen_loops.py"

print("MOE_ROOT", MOE_ROOT)
print("SOURCE_WORKSPACE", SOURCE_WORKSPACE)
print("PREV_MOE_WORKSPACE", PREV_MOE_WORKSPACE)
print("WORKSPACE", WORKSPACE)
print("RUNNER", RUNNER)

MOE_ROOT /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/moe
SOURCE_WORKSPACE /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/03_main_bn_residual_dqa_experiment
PREV_MOE_WORKSPACE /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/moe/output/05_router_ten_loops
WORKSPACE /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/moe/output/06_spatial_expert_fifteen_loops
RUNNER /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/moe/scripts/run_moe_06_spatial_expert_fifteen_loops.py


## Execute Fifteen Screening Loops

In [2]:
cmd = [
    sys.executable,
    str(RUNNER),
    "--workspace-root", str(WORKSPACE),
    "--source-workspace", str(SOURCE_WORKSPACE),
    "--prev-moe-workspace", str(PREV_MOE_WORKSPACE),
    "--notify",
]
print(" ".join(cmd))
subprocess.run(cmd, cwd=MOE_ROOT, check=True)

/opt/venv/bin/python3 /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/moe/scripts/run_moe_06_spatial_expert_fifteen_loops.py --workspace-root /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/moe/output/06_spatial_expert_fifteen_loops --source-workspace /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/03_main_bn_residual_dqa_experiment --prev-moe-workspace /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/moe/output/05_router_ten_loops --notify


DiscordNotifyResult(ok=True, chunks_sent=1, status_codes=(204,), dry_run=False, error=None)
Top 5 screened loops:
loop07_repair_shielded_local_experts projected= 0.204900 delta= 0.004900 confidence= high
loop14_negative_transfer_guard projected= 0.204350 delta= 0.004350 confidence= medium
loop03_fedmox_spatial_router projected= 0.202500 delta= 0.002500 confidence= medium
loop02_expert_choice_pseudogt_quota projected= 0.201457 delta= 0.001457 confidence= low
loop05_shared_private_ple_mmoe projected= 0.201400 delta= 0.001400 confidence= low
DiscordNotifyResult(ok=True, chunks_sent=1, status_codes=(204,), dry_run=False, error=None)


CompletedProcess(args=['/opt/venv/bin/python3', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/moe/scripts/run_moe_06_spatial_expert_fifteen_loops.py', '--workspace-root', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/moe/output/06_spatial_expert_fifteen_loops', '--source-workspace', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/03_main_bn_residual_dqa_experiment', '--prev-moe-workspace', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/moe/output/05_router_ten_loops', '--notify'], returncode=0)

## Split Evidence

In [3]:
split_evidence = pd.read_csv(WORKSPACE / "stats" / "06_spatial_expert_split_evidence.csv")
display(split_evidence)

,split,client,images,map50_95_dqa_aggregate,map50_95_warmup_repair,map50_95_dqa_repair,gain_dqa_vs_repair,repair_overwrite_loss,pseudo_boxes_kept,boxes_per_kept_image,mean_conf,mean_stability,mean_score
0,highway_day,client0_highway_day,1290.0,0.201,0.189,0.197,0.012,0.004,10787,7.2542,0.842009,0.942450,0.796646
1,highway_night,client1_highway_night,1007.0,0.132,0.125,0.127,0.007,0.005,8679,5.9609,0.778655,0.917807,0.718957
2,citystreet_day,client2_citystreet_day,3067.0,0.223,0.212,0.219,0.011,0.004,13501,9.0489,0.835234,0.939346,0.787885
3,citystreet_night,client3_citystreet_night,2582.0,0.155,0.144,0.152,0.011,0.003,11909,8.0412,0.771284,0.915274,0.710052
4,residential_day,client4_residential_day,862.0,0.241,0.224,0.235,0.017,0.006,9623,6.4627,0.860080,0.949911,0.820081
5,residential_night,client5_residential_night,279.0,0.160,0.157,0.151,0.003,0.009,8591,5.8008,0.828725,0.932196,0.775295


## Fifteen Loop Scoreboard

In [4]:
scoreboard = pd.read_csv(WORKSPACE / "stats" / "06_spatial_expert_scoreboard.csv")
display(
    scoreboard[
        [
            "loop_id",
            "rank_score",
            "screened_projected_map50_95",
            "screened_delta_map50_95",
            "confidence",
            "implementation_change",
            "rationale",
        ]
    ]
)

,loop_id,rank_score,screened_projected_map50_95,screened_delta_map50_95,confidence,implementation_change,rationale
0,loop07_repair_shielded_local_experts,0.207900,0.204900,0.004900,high,Run server repair on shared expert; keep local...,keeps observed BN/neck-head bonus 0.0010; reco...
1,loop14_negative_transfer_guard,0.205850,0.204350,0.004350,medium,Track split-proxy risk from pseudo stats; supp...,recovers part of repair overwrite 0.0030; guar...
2,loop03_fedmox_spatial_router,0.205500,0.202500,0.002500,medium,"Use K=4 local experts: easy-day, dense-scene, ...",keeps observed BN/neck-head bonus 0.0010; soft...
3,loop02_expert_choice_pseudogt_quota,0.204457,0.201457,0.001457,low,"For each expert, rank boxes by learnability=(s...",uses learnability spread 0.0014; class quota p...
4,loop05_shared_private_ple_mmoe,0.204400,0.201400,0.001400,low,"Update shared expert on all stable pseudo GT, ...",soft routing reduces hard pseudo-GT errors; sh...
5,loop10_confidence_free_learnability_gate,0.204300,0.201300,0.001300,low,Replace confidence threshold with learnability...,class quota preserves DQA classwise intent; cr...
6,loop06_bn_neck_local_expert,0.204000,0.201000,0.001000,low,Freeze backbone; train K=4 neck/head+BN local ...,keeps observed BN/neck-head bonus 0.0010
7,loop04_dataset_aware_local_moe,0.203850,0.202350,0.002350,medium,Use scene/day-night tag only as router prior; ...,soft routing reduces hard pseudo-GT errors; gu...
8,loop12_class_density_expert_choice,0.203457,0.201957,0.001957,medium,Each expert owns class-density buckets; apply ...,uses learnability spread 0.0014; class quota p...
9,loop01_soft_local_region_router,0.203300,0.201800,0.001800,medium,Create region tokens from bbox center/area/cla...,keeps observed BN/neck-head bonus 0.0010; soft...


## Explicit Fifteen-loop Trace

In [5]:
trace = pd.read_csv(WORKSPACE / "stats" / "06_spatial_expert_loop_trace.csv")
display(
    trace[
        [
            "loop_index",
            "loop_id",
            "step_1_research",
            "step_5_execution",
            "step_6_result_summary",
            "step_7_next_direction",
        ]
    ]
)

,loop_index,loop_id,step_1_research,step_5_execution,step_6_result_summary,step_7_next_direction
0,1,loop01_soft_local_region_router,Soft MoE; PSSFL/FedMox,executed as fast evidence-screening loop using...,"projected mAP50:95=0.201800, delta=0.001800, c...",keep as ablation after the selected full run
1,2,loop02_expert_choice_pseudogt_quota,Expert Choice Routing,executed as fast evidence-screening loop using...,"projected mAP50:95=0.201457, delta=0.001457, c...",keep as ablation after the selected full run
2,3,loop03_fedmox_spatial_router,PSSFL/FedMox,executed as fast evidence-screening loop using...,"projected mAP50:95=0.202500, delta=0.002500, c...",keep as ablation after the selected full run
3,4,loop04_dataset_aware_local_moe,DAMEX,executed as fast evidence-screening loop using...,"projected mAP50:95=0.202350, delta=0.002350, c...",keep as ablation after the selected full run
4,5,loop05_shared_private_ple_mmoe,MMoE; PLE,executed as fast evidence-screening loop using...,"projected mAP50:95=0.201400, delta=0.001400, c...",keep as ablation after the selected full run
5,6,loop06_bn_neck_local_expert,FedBN; previous MoE 05,executed as fast evidence-screening loop using...,"projected mAP50:95=0.201000, delta=0.001000, c...",keep as ablation after the selected full run
6,7,loop07_repair_shielded_local_experts,PSSFL Soft-Mixture; previous 03,executed as fast evidence-screening loop using...,"projected mAP50:95=0.204900, delta=0.004900, c...",promote to full experiment
7,8,loop08_entropy_load_balanced_router,Switch/GShard load balance; Expert Choice,executed as fast evidence-screening loop using...,"projected mAP50:95=0.200557, delta=0.000557, c...",keep as ablation after the selected full run
8,9,loop09_dense_to_sparse_curriculum,Soft MoE; Switch,executed as fast evidence-screening loop using...,"projected mAP50:95=0.201600, delta=0.001600, c...",keep as ablation after the selected full run
9,10,loop10_confidence_free_learnability_gate,Semi-supervised detection uncertainty filtering,executed as fast evidence-screening loop using...,"projected mAP50:95=0.201300, delta=0.001300, c...",keep as ablation after the selected full run


## Selected Full Experiment Candidate

In [6]:
import json

candidate_path = WORKSPACE / "stats" / "06_selected_full_experiment_candidate.json"
candidate = json.loads(candidate_path.read_text(encoding="utf-8"))
print(json.dumps(candidate, indent=2, ensure_ascii=False))

{
  "created_utc": "2026-05-08T23:02:13.533090+00:00",
  "protocol": "scene_daynight_dqa_moe_06_spatial_expert_fifteen_loops_v1",
  "selected_loop": "loop07_repair_shielded_local_experts",
  "selected_hypothesis": "Server repair is useful, but it overwrote DQA gains; repair only the shared path and keep local experts un-repaired.",
  "selected_implementation_change": "Run server repair on shared expert; keep local residual experts as frozen add-ons during final evaluation.",
  "source_real_anchor": {
    "dqa_aggregate_map50_95": 0.2,
    "dqa_repair_map50_95": 0.195,
    "warmup_server_repair_map50_95": 0.186,
    "previous_moe_best_map50_95": 0.201
  },
  "full_experiment_plan": {
    "name": "07_local_region_expert_dqa_full",
    "K": 4,
    "phase1_rounds": 30,
    "phase2_rounds": 2,
    "freeze_backbone": true,
    "trainable_parts": [
      "neck",
      "head",
      "batch_norm"
    ],
    "expert_units": [
      "shared_source_repair",
      "easy_day_stable_regions",
      "

## Markdown Report

In [7]:
report = WORKSPACE / "06_spatial_expert_fifteen_loop_report.md"
print(report)
print(report.read_text(encoding="utf-8")[:7000])

/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/moe/output/06_spatial_expert_fifteen_loops/06_spatial_expert_fifteen_loop_report.md
# MoE x DQA 06: Local-Region Expert Fifteen Loops

- created_utc: 2026-05-08T23:02:13.533244+00:00
- protocol: scene_daynight_dqa_moe_06_spatial_expert_fifteen_loops_v1
- mode: fast screening over 15 local-region expert hypotheses; not 15 full YOLO trainings

## Evidence Used

- warmup + server repair mAP50:95: 0.186
- BN-residual DQA aggregate mAP50:95: 0.200
- BN-residual DQA + server repair mAP50:95: 0.195
- previous MoE best mAP50:95: 0.201
- observed repair overwrite loss: 0.005

## Top Loop Ranking

| rank | loop | rank score | projected mAP50:95 | delta | confidence | implementation |
|---:|---|---:|---:|---:|---|---|
| 1 | loop07_repair_shielded_local_experts | 0.207900 | 0.204900 | 0.004900 | high | Run server repair on shared expert; keep local residual experts as frozen add-ons during final evaluation. |
| 2 